# Two-stage nested-CV pipeline (DMP)

This notebook compares filter-based feature selection followed by embedded feature selection. Both stages are fitted using training data only inside nested cross-validation.


## 0. Reproducibility and expected input

Expected CSV format:

- one row per sample;
- one column per upstream DEG/DMP feature;
- one binary target column named `label` by default (`0=normal`, `1=CC`);
- no sample identifiers among the feature columns unless explicitly removed before running.

For a public repository, keep raw/private data outside Git. Use relative paths and document how the processed input matrix was generated.


## Important: select exactly one Stage-1 method before each run

Set `STAGE1_METHODS` in the next configuration cell to exactly one of `['pearson']`, `['relieff']`, or `['mrmr']`. Run the notebook from the configuration cell downward after changing it. Do not redefine `STAGE1_METHODS` in a later cell.

`stage1_methods` and the complete classifier grids are included in `RUN_CONFIG`, which is hashed to create the checkpoint ID. Therefore each feature-selection method receives a different checkpoint directory and must not be combined at fold level with another method's checkpoint. The configuration is also saved inside that checkpoint directory for permanent provenance.

Known mapping: DEG Pearson Stage 1 = `project/results/machine-learning/current/deg/checkpoints/28ffd9776038/` (160 completed outer folds: four ratios × eight classifiers × five folds).

Each newly completed outer fold appends its wall-clock timing to `runtime_log.csv` in the hashed checkpoint directory. `runtime_summary.csv` is refreshed after every successful fold with classifier-level totals and per-fold mean/minimum/maximum runtime. Failed folds remain in `failures.csv` and are not counted as completed runtime.


In [ ]:
from __future__ import annotations

import json, os, platform, sys, time, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

try:
    from skrebate import ReliefF
except ImportError as e:
    raise ImportError('Install skrebate: pip install skrebate') from e
try:
    from mrmr import mrmr_classif
except ImportError as e:
    raise ImportError('Install mrmr-selection: pip install mrmr-selection') from e

warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
# ---------------------------
# USER CONFIGURATION
# ---------------------------
TEST_MODE = False  # Set to True for a short pipeline check.
DATASET_NAME = 'DMP'  # 'DEG' or 'DMP' (used only in output names)
PROJECT_ROOT = Path.cwd()  # Run the notebook from the repository root.
INPUT_CSV = PROJECT_ROOT / "project/data/differential-analysis/DMP/3203_DMPs_Mval_ComBat_corrected_with_label.csv"
LABEL_COL = 'label'
OUTPUT_ROOT = PROJECT_ROOT / "project/results/machine-learning/current" / DATASET_NAME.lower()

# Full analysis agreed for manuscript
FULL_OUTER_FOLDS = 5
FULL_INNER_FOLDS = 5
FULL_REPEATS = 1  # one nested-CV execution; no repeated nested CV
FULL_RATIOS = (0.2, 0.4, 0.6, 0.8)




BASE_SEED = 42
N_JOBS_GRID = 1

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Output:', OUTPUT_ROOT.resolve())


## 1. Run configuration

Full mode uses the supervisor-approved **single 5 × 5 nested-CV execution**, four retention ratios (0.2, 0.4, 0.6 and 0.8), and the reduced classifier grids recorded in the pipeline decisions. Test mode deliberately shrinks folds, ratios, classifiers and hyperparameter grids so that bugs are found before a full run.

checkpoint → protects completed outer-fold RESULTS

cache      → avoids recomputing identical transformers


In [ ]:

OUTER_FOLDS, INNER_FOLDS, N_REPEATS = FULL_OUTER_FOLDS, FULL_INNER_FOLDS, FULL_REPEATS
RATIOS = FULL_RATIOS
CLASSIFIER_NAMES = [
    'Decision Tree', 'Gradient Boosting', 'KNN', 'XGBoost',
    'Logistic Regression', 'Naive Bayes', 'Random Forest', 'SVM'
]

# Select exactly one method for this run. Change only this line between Stage-1 runs.
STAGE1_METHODS = ['mrmr']  # alternatives: ['pearson'] or ['relieff']
VALID_STAGE1_METHODS = {'pearson', 'relieff', 'mrmr'}
if len(STAGE1_METHODS) != 1 or STAGE1_METHODS[0] not in VALID_STAGE1_METHODS:
    raise ValueError("STAGE1_METHODS must contain exactly one of: pearson, relieff, mrmr")
FULL_GRIDS = {
    'Random Forest': {'clf__n_estimators':[100,300], 'clf__max_depth':[None,10,20], 'clf__min_samples_split':[2,5]},
    'SVM': {'clf__C':[0.1,1,10], 'clf__gamma':['scale','auto',0.001,0.01], 'clf__kernel':['rbf']},
    'Logistic Regression': {'clf__C':[0.1,1,10,100], 'clf__solver':['liblinear','lbfgs'], 'clf__penalty':['l2']},
    'Naive Bayes': {'clf__var_smoothing':[1e-9,1e-8,1e-7,1e-6]},
    'Decision Tree': {'clf__max_depth':[None,5,10], 'clf__min_samples_split':[2,5], 'clf__min_samples_leaf':[1,2]},
    'KNN': {'clf__n_neighbors':[3,5,7,9], 'clf__metric':['euclidean','manhattan']},
    'Gradient Boosting': {'clf__n_estimators':[100,200], 'clf__learning_rate':[0.05,0.1], 'clf__max_depth':[2,3,5]},
    'XGBoost': {'clf__n_estimators':[100,200], 'clf__learning_rate':[0.05,0.1], 'clf__max_depth':[2,3,5]},
}
SELECTOR_CODE_VERSION = "relieff-retention-v2"
PARAM_GRIDS = FULL_GRIDS
RUN_CONFIG = {
    'dataset_name': DATASET_NAME,
    'input_csv': str(INPUT_CSV), 'label_col': LABEL_COL,
    'outer_folds': OUTER_FOLDS, 'inner_folds': INNER_FOLDS,
    'repeats': N_REPEATS, 'ratios': list(RATIOS),
    'stage1_methods': STAGE1_METHODS,
    'param_grids': PARAM_GRIDS,
    'classifiers': CLASSIFIER_NAMES, 'base_seed': BASE_SEED,
    'n_jobs_grid': N_JOBS_GRID,
    'stage1_selection_rule': 'highest_mean_outer_fold_auprc',
    'selector_code_version': SELECTOR_CODE_VERSION
}
(OUTPUT_ROOT / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2))
RUN_CONFIG


In [ ]:
# ---------------------------
# CHECKPOINT / RESUME CONFIG
# ---------------------------

import hashlib
from datetime import datetime

# Create a reproducible signature for this exact experimental setup.
# If TEST_MODE, folds, ratios, classifiers, dataset, etc. change,
# a different checkpoint directory will be used.
run_config_string = json.dumps(RUN_CONFIG, sort_keys=True)
RUN_ID = hashlib.sha256(
    run_config_string.encode("utf-8")
).hexdigest()[:12]

CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / RUN_ID
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
(CHECKPOINT_ROOT / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2))

CHECKPOINT_METRICS = CHECKPOINT_ROOT / "fold_metrics.csv"
CHECKPOINT_PREDS = CHECKPOINT_ROOT / "fold_predictions.csv"
CHECKPOINT_FEATURES = CHECKPOINT_ROOT / "selected_features.jsonl"
CHECKPOINT_RUNTIME = CHECKPOINT_ROOT / "runtime_log.csv"
RUNTIME_SUMMARY = CHECKPOINT_ROOT / "runtime_summary.csv"
FAILURE_LOG = CHECKPOINT_ROOT / "failures.csv"

print("Run ID:", RUN_ID)
print("Checkpoint directory:", CHECKPOINT_ROOT.resolve())


In [ ]:
# ---------------------------
# CHECKPOINT HELPER FUNCTIONS
# ---------------------------

def make_fold_key(stage, method, ratio, classifier, repeat, outer_fold):
    """
    Unique identity for one completed outer-CV fold.
    """
    return (
        str(stage),
        str(method),
        float(ratio),
        str(classifier),
        int(repeat),
        int(outer_fold),
    )


def load_completed_folds():
    """
    Read successful outer-fold evaluations from the checkpoint metrics file.

    Only folds present in fold_metrics.csv are considered completed.
    Failed folds are NOT considered completed and can therefore be retried.
    """
    if not CHECKPOINT_METRICS.exists():
        return set()

    df = pd.read_csv(CHECKPOINT_METRICS)

    required_cols = {
        'stage',
        'method',
        'ratio',
        'classifier',
        'repeat',
        'outer_fold'
    }

    if not required_cols.issubset(df.columns):
        raise ValueError(
            "Checkpoint metrics file exists but does not contain "
            f"the required columns: {required_cols}"
        )

    completed = set()

    for _, row in df.iterrows():
        key = make_fold_key(
            row['stage'],
            row['method'],
            row['ratio'],
            row['classifier'],
            row['repeat'],
            row['outer_fold']
        )
        completed.add(key)

    return completed


def append_dataframe_checkpoint(df, path):
    """
    Append rows to a CSV checkpoint.

    Writes the header only when the file does not already exist.
    """
    if df is None or len(df) == 0:
        return

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        path,
        mode='a',
        header=not path.exists(),
        index=False
    )


def append_jsonl_checkpoint(record, path):
    """
    Append one JSON record as one line.
    Useful for selected-feature information.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, 'a') as f:
        f.write(json.dumps(record, default=str) + '\n')


In [ ]:
def append_runtime_checkpoint(runtime_record):
    """Append one successful outer-fold timing record and refresh its summary."""
    runtime_path = CHECKPOINT_ROOT / 'runtime_log.csv'
    summary_path = CHECKPOINT_ROOT / 'runtime_summary.csv'
    append_dataframe_checkpoint(pd.DataFrame([runtime_record]), runtime_path)

    runtime_df = pd.read_csv(runtime_path)
    summary = (
        runtime_df.groupby(
            ['run_id', 'dataset', 'stage', 'fs_method', 'ratio', 'classifier'],
            as_index=False,
            dropna=False
        )
        .agg(
            completed_outer_folds=('elapsed_seconds', 'count'),
            total_seconds=('elapsed_seconds', 'sum'),
            mean_seconds_per_fold=('elapsed_seconds', 'mean'),
            min_seconds_per_fold=('elapsed_seconds', 'min'),
            max_seconds_per_fold=('elapsed_seconds', 'max')
        )
    )
    summary['total_minutes'] = summary['total_seconds'] / 60
    summary['total_hours'] = summary['total_seconds'] / 3600
    summary.to_csv(summary_path, index=False)
    return summary


In [ ]:
# ---------------------------
# FAILURE LOGGING
# ---------------------------

import traceback


def log_failure(
    stage,
    method,
    ratio,
    classifier,
    repeat,
    outer_fold,
    exception
):
    """
    Record one failed outer-fold evaluation.

    A failure is logged but is NOT marked as completed,
    so it can be retried when the pipeline is resumed.
    """

    failure_record = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'stage': str(stage),
        'method': str(method),
        'ratio': float(ratio),
        'classifier': str(classifier),
        'repeat': int(repeat),
        'outer_fold': int(outer_fold),
        'error_type': type(exception).__name__,
        'error_message': str(exception),
        'traceback': traceback.format_exc()
    }

    append_dataframe_checkpoint(
        pd.DataFrame([failure_record]),
        FAILURE_LOG
    )

    print(
        f"[FAILED] "
        f"{stage} | {method} | ratio={ratio} | "
        f"{classifier} | repeat={repeat} | fold={outer_fold}"
    )

    print(
        f"         {type(exception).__name__}: {exception}"
    )


In [ ]:
from joblib import Memory

ML_CACHE_DIR = (
    OUTPUT_ROOT
    / "sklearn_cache"
    / RUN_ID
)

ML_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ML_MEMORY = Memory(
    location=ML_CACHE_DIR,
    verbose=0,
)


print("Pipeline cache:", ML_CACHE_DIR.resolve())


## 2. Load and validate data

For a GitHub release, do not hard-code `/Users/...` paths. Put the processed matrix under a documented local `data/` path or pass another relative path above.


In [ ]:
input_check = pd.read_csv(INPUT_CSV)

print("Shape:", input_check.shape)
print("First columns:", input_check.columns[:10].tolist())
print("Last columns:", input_check.columns[-10:].tolist())


In [ ]:
METADATA_COLS = ['sample_id', 'batch', 'condition']

def load_dataset(path, label_col):
    df = pd.read_csv(path)

    y = df[label_col].copy()

    X = df.drop(
        columns=[label_col] + METADATA_COLS
    ).copy()

    # Safety check: ML features must all be numeric
    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()

    if non_numeric:
        raise ValueError(
            f"Non-numeric columns remain in X: {non_numeric}"
        )

    print(f"Samples: {X.shape[0]}")
    print(f"ML features: {X.shape[1]}")
    print(f"Class distribution: {y.value_counts().to_dict()}")

    return X, y

# Uncomment after INPUT_CSV is set correctly.
X, y = load_dataset(INPUT_CSV, LABEL_COL)
print('X:', X.shape, 'class counts:', y.value_counts().to_dict())


## 3. Feature selectors

All selectors are sklearn-compatible transformers so they can be fitted only on the training partition.

- **Pearson:** rank by absolute Pearson correlation with the training labels.
- **ReliefF:** rank using ReliefF fitted on training data.
- **mRMR:** obtain the ordered mRMR feature list using training data.
- **LASSO:** L1-logistic regression; remove zero coefficients, rank remaining features by absolute coefficient, then apply the retention ratio to that non-zero pool.
- **Elastic Net:** same logic with elastic-net logistic regression.
- **RF:** rank by training-fitted RF feature importance and retain the requested ratio.

For LASSO/Elastic Net, selector hyperparameters are configurable. The defaults below are deliberately explicit and reproducible; if they are tuned, that tuning must itself remain inside the relevant training data.


## 4. Classifiers and hyperparameter grids

SMOTE stays **inside** the pipeline. A scaler is included for distance/coefficient-sensitive classifiers and is also training-fitted. Test mode uses deliberately tiny grids; full mode uses the manuscript grids.


In [ ]:
# ---------------------------------------
# SHARED ML IMPLEMENTATION IMPORTS
# ---------------------------------------

import sys
from pathlib import Path

def find_pipeline_root():
    """
    Find current-pipeline regardless of whether the notebook kernel
    starts from the repository root, current-pipeline, DEG, or DMP.
    """
    start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        selector_file = candidate / "ml_selectors.py"
        workflow_file = candidate / "stage2_workflow.py"

        if selector_file.exists() and workflow_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find current-pipeline containing both "
        "ml_selectors.py and stage2_workflow.py."
    )


PIPELINE_ROOT = find_pipeline_root()

if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))


from ml_selectors import (
    RankedSelector,
    SequentialSelector,
)

from stage2_workflow import (
    AtomicStage2Checkpoint,
    Stage2Spec,
    evaluate_stage2_configuration,
    stage2_run_id,
    summarize_stage2,
)


print("Pipeline root:", PIPELINE_ROOT)
print("Selector source:", Path(sys.modules["ml_selectors"].__file__).resolve())
print(
    "Stage 2 workflow source:",
    Path(sys.modules["stage2_workflow"].__file__).resolve(),
)


In [ ]:
def make_classifier_pipeline(name: str, seed: int) -> Pipeline:
    smote_k = 3  # conservative for small inner-training minority classes
    if name == 'Random Forest':
        steps = [('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', RandomForestClassifier(random_state=seed, n_jobs=1))]
    elif name == 'SVM':
        steps = [('scale', StandardScaler()), ('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', SVC(probability=True, random_state=seed))]
    elif name == 'Logistic Regression':
        steps = [('scale', StandardScaler()), ('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', LogisticRegression(max_iter=3000, random_state=seed))]
    elif name == 'Naive Bayes':
        steps = [ ('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', GaussianNB())]
    elif name == 'Decision Tree':
        steps = [('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', DecisionTreeClassifier(random_state=seed))]
    elif name == 'KNN':
        steps = [('scale', StandardScaler()), ('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', KNeighborsClassifier(weights='distance'))]
    elif name == 'Gradient Boosting':
        steps = [('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', GradientBoostingClassifier(random_state=seed))]
    elif name == 'XGBoost':
        steps = [('smote', SMOTE(k_neighbors=smote_k, random_state=seed)),
                 ('clf', XGBClassifier(eval_metric='logloss', random_state=seed, n_jobs=1))]
    else:
        raise ValueError(name)
    return Pipeline(steps)



PARAM_GRIDS = FULL_GRIDS


## 5. Core nested-CV evaluator

Each `(FS method, retention ratio, classifier)` is treated as a **predefined pipeline to benchmark**. The inner CV tunes the classifier hyperparameters using AUPRC. The outer fold evaluates that pipeline. This is different from an AutoML-style inner loop that is allowed to switch freely among FS methods and ratios.

For each outer-training refit we save:
- selected feature names;
- number of retained features;
- chosen classifier hyperparameters;
- outer-fold predictions and metrics.


In [ ]:
def scores_from_estimator(estimator, X):
    if hasattr(estimator, 'predict_proba'):
        return estimator.predict_proba(X)[:, 1]
    return estimator.decision_function(X)

def metric_row(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        'auprc': average_precision_score(y_true, y_score),
        'roc_auc': roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall_sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn/(tn+fp) if (tn+fp) else np.nan,
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }

def evaluate_predefined_pipeline(X, y, fs_method, ratio, classifier_name,
                                 stage1_rule=None, output_dir=None):
    rows, selected_records, pred_records = [], [], []
    t0 = time.time()

    for repeat in range(N_REPEATS):
        outer_seed = BASE_SEED + repeat * 1000
        outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=outer_seed)

        for outer_fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
            Xtr, Xte = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
            ytr, yte = y.iloc[tr_idx].copy(), y.iloc[te_idx].copy()
            seed = outer_seed + outer_fold

            # Stage-2 first filter is refitted on OUTER TRAINING only. It is never a globally fixed gene list.
            if stage1_rule is not None:
                f_method, f_ratio = stage1_rule
                first = RankedSelector(f_method, f_ratio, random_state=seed).fit(Xtr, ytr)
                Xtr_work = first.transform(Xtr)
                Xte_work = first.transform(Xte)
                first_features = first.selected_features_
            else:
                Xtr_work, Xte_work = Xtr, Xte
                first_features = None

            # Current FS is fitted on outer-training for the outer evaluation.
            selector = RankedSelector(fs_method, ratio, random_state=seed).fit(Xtr_work, ytr)
            Xtr_sel = selector.transform(Xtr_work)
            Xte_sel = selector.transform(Xte_work)

            # Inner CV tunes classifier only, using AUPRC.
            inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed + 100)
            pipe = make_classifier_pipeline(classifier_name, seed)
            grid = GridSearchCV(
                pipe, PARAM_GRIDS[classifier_name], scoring='average_precision',
                cv=inner_cv, n_jobs=N_JOBS_GRID, refit=True, error_score='raise'
            )
            grid.fit(Xtr_sel, ytr)

            y_score = scores_from_estimator(grid.best_estimator_, Xte_sel)
            y_pred = grid.best_estimator_.predict(Xte_sel)
            metrics = metric_row(yte, y_pred, y_score)
            rows.append({
                'repeat': repeat+1, 'outer_fold': outer_fold,
                'fs_method': fs_method, 'ratio': ratio,
                'classifier': classifier_name,
                'n_input_features': X.shape[1],
                'n_stage1_features': len(first_features) if first_features is not None else np.nan,
                'n_candidate_features': selector.n_candidate_features_,
                'n_retained_features': selector.n_selected_features_,
                'best_inner_auprc': grid.best_score_,
                'best_params': json.dumps(grid.best_params_, sort_keys=True),
                **metrics,
            })
            selected_records.append({
                'repeat': repeat+1, 'outer_fold': outer_fold,
                'fs_method': fs_method, 'ratio': ratio, 'classifier': classifier_name,
                'selected_features': selector.selected_features_,
                'stage1_features': first_features,
            })
            for idx, truth, pred, score in zip(te_idx, yte, y_pred, y_score):
                pred_records.append({
                    'sample_index': int(idx), 'repeat': repeat+1, 'outer_fold': outer_fold,
                    'fs_method': fs_method, 'ratio': ratio, 'classifier': classifier_name,
                    'y_true': int(truth), 'y_pred': int(pred), 'y_score': float(score)
                })

    metrics_df = pd.DataFrame(rows)
    preds_df = pd.DataFrame(pred_records)
    elapsed = time.time() - t0
    print(f'{fs_method} {ratio} | {classifier_name}: {elapsed:.1f}s')
    return metrics_df, selected_records, preds_df


### Why Stage 2 needs an extra inner-fold note

The function above is a **repository-ready baseline for the agreed predefined-pipeline benchmark**, with the Stage-1 filter refitted on each outer-training set. For the strictest leakage boundary during Stage-2 classifier tuning, the Stage-1 and Stage-2 selectors should ultimately be represented as transformers *inside the inner `GridSearchCV` pipeline* so they are refitted again for each inner-training split.

The full-run helper below uses that stricter integrated form. This matters because Stage 2 contains two supervised selection operations.


## full pipeline


In [ ]:
import time
def make_full_pipeline(classifier_name, selector, seed):
    # FS occurs before SMOTE so synthetic observations never influence feature selection.
    base = make_classifier_pipeline(classifier_name, seed)
    return Pipeline([('fs', selector)] + list(base.steps), memory=ML_MEMORY)

def evaluate_integrated_pipeline(
    X,
    y,
    selector_factory,
    fs_label,
    ratio,
    classifier_name,
    stage
):
    """
    Strict nested CV with fold-level checkpointing.

    - Feature selection remains inside GridSearchCV.
    - Successful outer folds are checkpointed immediately.
    - Completed folds are skipped when resuming.
    - Failed folds are logged and the benchmark continues.
    """

    rows = []
    selected_records = []
    pred_records = []

    # Load folds that have already completed successfully
    completed_folds = load_completed_folds()

    for repeat in range(N_REPEATS):

        outer_seed = BASE_SEED + repeat * 1000
        outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=outer_seed)

        for outer_fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y),start=1):

            # ---------------------------------
            # UNIQUE IDENTITY OF THIS OUTER FOLD
            # ---------------------------------

            fold_key = make_fold_key(
                stage=stage,
                method=fs_label,
                ratio=ratio,
                classifier=classifier_name,
                repeat=repeat + 1,
                outer_fold=outer_fold
            )

            # ---------------------------------
            # RESUME: SKIP COMPLETED FOLDS
            # ---------------------------------

            if fold_key in completed_folds:

                print(
                    f"[SKIP] {stage} | {fs_label} | "
                    f"ratio={ratio} | {classifier_name} | "
                    f"repeat={repeat+1} | fold={outer_fold}"
                )

                continue

            print(
                f"    [RUN] repeat={repeat+1}/{N_REPEATS} "
                f"fold={outer_fold}/{OUTER_FOLDS}"
            )

            try:

                # -----------------------------
                # OUTER TRAIN / TEST SPLIT
                # -----------------------------

                Xtr = X.iloc[tr_idx].copy()
                Xte = X.iloc[te_idx].copy()

                ytr = y.iloc[tr_idx].copy()
                yte = y.iloc[te_idx].copy()

                seed = outer_seed + outer_fold

                # -----------------------------
                # BUILD PIPELINE
                # -----------------------------

                selector = selector_factory(seed)

                pipe = make_full_pipeline(
                    classifier_name,
                    selector,
                    seed
                )

                inner_cv = StratifiedKFold(
                    n_splits=INNER_FOLDS,
                    shuffle=True,
                    random_state=seed + 100
                )

                grid = GridSearchCV(
                    pipe,
                    PARAM_GRIDS[classifier_name],
                    scoring='average_precision',
                    cv=inner_cv,
                    n_jobs=N_JOBS_GRID,
                    refit=True,
                    error_score='raise'
                )

                # -----------------------------
                # INNER CV + REFIT
                # -----------------------------
                fold_started_at = datetime.now().astimezone()
                fold_start_time = time.perf_counter()
                grid.fit(Xtr, ytr)

                best = grid.best_estimator_

                # -----------------------------
                # OUTER TEST EVALUATION
                # -----------------------------

                y_score = scores_from_estimator(best, Xte)
                y_pred = best.predict(Xte)
                fold_elapsed_seconds = time.perf_counter() - fold_start_time
                fold_finished_at = datetime.now().astimezone()
                

                fs = best.named_steps['fs']

                selected = list(fs.selected_features_)

                n_stage1 = (
                    len(fs.stage1_features_)
                    if hasattr(fs, 'stage1_features_')
                    else np.nan
                )

                metrics = metric_row(
                    yte,
                    y_pred,
                    y_score
                )

                # -----------------------------
                # METRIC RECORD
                # -----------------------------

                metric_record = {
                    'stage': stage,
                    'repeat': repeat + 1,
                    'outer_fold': outer_fold,
                    'fs_method': fs_label,
                    'method': fs_label,
                    'ratio': ratio,
                    'classifier': classifier_name,
                    'n_input_features': X.shape[1],
                    'n_stage1_features': n_stage1,
                    'n_retained_features': len(selected),
                    'best_inner_auprc': grid.best_score_,
                    'best_params': json.dumps(
                        grid.best_params_,
                        sort_keys=True
                    ),
                    'runtime_seconds': fold_elapsed_seconds,
                    'runtime_minutes': fold_elapsed_seconds / 60,
                    **metrics
                }

                # -----------------------------
                # SELECTED FEATURES RECORD
                # -----------------------------

                feature_record = {
                    'stage': stage,
                    'repeat': repeat + 1,
                    'outer_fold': outer_fold,
                    'fs_method': fs_label,
                    'method': fs_label,
                    'ratio': ratio,
                    'classifier': classifier_name,
                    'selected_features': selected
                }

                # -----------------------------
                # PREDICTION RECORDS
                # -----------------------------

                fold_pred_records = []

                for idx, truth, pred, score in zip(te_idx,yte,y_pred,y_score):
                    fold_pred_records.append({
                        'stage': stage,
                        'sample_index': int(idx),
                        'repeat': repeat + 1,
                        'outer_fold': outer_fold,
                        'fs_method': fs_label,
                        'method': fs_label,
                        'ratio': ratio,
                        'classifier': classifier_name,
                        'y_true': int(truth),
                        'y_pred': int(pred),
                        'y_score': float(score)
                    })

                # =================================
                # CHECKPOINT IMMEDIATELY
                # =================================

                append_dataframe_checkpoint(
                    pd.DataFrame([metric_record]),
                    CHECKPOINT_METRICS
                )

                append_dataframe_checkpoint(
                    pd.DataFrame(fold_pred_records),
                    CHECKPOINT_PREDS
                )

                append_jsonl_checkpoint(
                    feature_record,
                    CHECKPOINT_FEATURES
                )

                runtime_record = {
                    'timestamp_start': fold_started_at.isoformat(timespec='seconds'),
                    'timestamp_end': fold_finished_at.isoformat(timespec='seconds'),
                    'elapsed_seconds': fold_elapsed_seconds,
                    'dataset': DATASET_NAME,
                    'stage': stage,
                    'fs_method': fs_label,
                    'ratio': ratio,
                    'classifier': classifier_name,
                    'repeat': repeat + 1,
                    'outer_fold': outer_fold,
                    'status': 'success',
                    'run_id': RUN_ID
                }
                append_runtime_checkpoint(runtime_record)

                # Also keep in memory for normal return
                rows.append(metric_record)
                selected_records.append(feature_record)
                pred_records.extend(fold_pred_records)

                # Mark successful in current session
                completed_folds.add(fold_key)

                print(
                    f"    [DONE] repeat={repeat+1} "
                    f"fold={outer_fold} "
                    f"AUPRC={metrics['auprc']:.4f} "
                    f"time={fold_elapsed_seconds / 60:.2f} min"
                )

            except Exception as e:

                # =================================
                # FAILURE DOES NOT KILL BENCHMARK
                # =================================

                log_failure(
                    stage=stage,
                    method=fs_label,
                    ratio=ratio,
                    classifier=classifier_name,
                    repeat=repeat + 1,
                    outer_fold=outer_fold,
                    exception=e
                )

                # Continue with next outer fold
                continue

    return (
        pd.DataFrame(rows),
        selected_records,
        pd.DataFrame(pred_records)
    )


In [ ]:
# ---------------------------
# CHECKPOINT RESULT LOADERS
# ---------------------------

def load_checkpoint_metrics(stage=None):
    """
    Load all successfully checkpointed outer-fold metrics
    for the current RUN_ID.
    """
    if not CHECKPOINT_METRICS.exists():
        return pd.DataFrame()

    df = pd.read_csv(CHECKPOINT_METRICS)

    if stage is not None:
        df = df[df['stage'] == stage].copy()

    return df.reset_index(drop=True)


def load_checkpoint_predictions(stage=None):
    """
    Load all successfully checkpointed outer-fold predictions
    for the current RUN_ID.
    """
    if not CHECKPOINT_PREDS.exists():
        return pd.DataFrame()

    df = pd.read_csv(CHECKPOINT_PREDS)

    if stage is not None:
        df = df[df['stage'] == stage].copy()

    return df.reset_index(drop=True)


def load_checkpoint_features(stage=None):
    """
    Load selected-feature records from JSONL checkpoint.
    """
    if not CHECKPOINT_FEATURES.exists():
        return []

    records = []

    with open(CHECKPOINT_FEATURES, 'r') as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if stage is None or record['stage'] == stage:
                records.append(record)

    return records


## 6. Stability functions

We record both:

- **individual feature selection frequency** across outer-training fits;
- **Nogueira-style chance-corrected stability** for a selector/configuration.

Selection frequency is not used as an extra Stage-1→Stage-2 cutoff. Inspect its distribution first.


In [ ]:
def selection_frequency(selected_records, all_features):
    counts = Counter()
    n = len(selected_records)
    for rec in selected_records:
        counts.update(set(rec['selected_features']))
    out = pd.DataFrame({'feature': list(all_features)})
    out['selected_count'] = out['feature'].map(counts).fillna(0).astype(int)
    out['selection_frequency'] = out['selected_count'] / max(n, 1)
    return out.sort_values(['selection_frequency','feature'], ascending=[False, True]).reset_index(drop=True)


def nogueira_stability(selected_records, all_features):
    """Chance-corrected stability based on the selection-indicator matrix.

    Returns NaN for degenerate cases. Appropriate for comparing selectors with different set sizes.
    """
    features = list(all_features)
    p = len(features); m = len(selected_records)
    if m < 2 or p < 2:
        return np.nan
    idx = {f:j for j,f in enumerate(features)}
    Z = np.zeros((m,p), dtype=float)
    for i,rec in enumerate(selected_records):
        for f in set(rec['selected_features']):
            if f in idx: Z[i,idx[f]] = 1.0
    kbar = Z.sum(axis=1).mean()
    if kbar <= 0 or kbar >= p:
        return np.nan
    # Sample variance of each feature's selection indicator.
    var = Z.var(axis=0, ddof=1)
    denom = (kbar/p) * (1 - kbar/p)
    return float(1 - var.mean()/denom) if denom > 0 else np.nan


## 7. Stage 1 runner: Pearson / ReliefF / mRMR

This produces the manuscript-style table dimensions:

`FS method × ratio × classifier`, plus retained-feature count and FS stability.

The **strict integrated evaluator** is used so FS is refitted inside every inner-training fold.


In [ ]:
def run_stage1(X, y, methods):
    """
    Stage 1:
    Filter-based feature selection benchmark.

    Supports fold-level checkpoint/resume.
    """

    record_map = {}

    for method in methods:
        for ratio in RATIOS:
            for clf in CLASSIFIER_NAMES:
                print(
                    f"[Stage 1] "
                    f"{method} ratio={ratio} clf={clf}"
                )

                factory = (
                    lambda seed, m=method, r=ratio:
                    RankedSelector(m,r,random_state=seed)
                )

                # evaluate_integrated_pipeline handles:
                # - completed-fold skipping
                # - immediate checkpointing
                # - failure logging
                evaluate_integrated_pipeline(X,y,factory,method,ratio,clf,stage='stage1')

    # -----------------------------------------
    # RECONSTRUCT COMPLETE RESULTS FROM DISK
    # -----------------------------------------

    all_metrics = load_checkpoint_metrics(stage='stage1')
    all_preds = load_checkpoint_predictions(stage='stage1')
    feature_records = load_checkpoint_features(stage='stage1')

    # Rebuild the record_map expected by
    # summarize_benchmark()
    for record in feature_records:

        key = (
            record['method'],
            float(record['ratio']),
            record['classifier']
        )

        if key not in record_map:
            record_map[key] = []

        record_map[key].append(record)

    return (
        all_metrics,
        record_map,
        all_preds
    )

def summarize_benchmark(metrics_df, record_map, all_features):
    rows=[]
    for key, g in metrics_df.groupby(['fs_method','ratio','classifier']):
        method, ratio, clf = key
        recs = record_map[key]
        rows.append({
            'fs_method':method, 'ratio':ratio, 'classifier':clf,
            'auprc_mean':g['auprc'].mean(), 'auprc_sd':g['auprc'].std(ddof=1),
            'retained_features_mean':g['n_retained_features'].mean(),
            'retained_features_sd':g['n_retained_features'].std(ddof=1),
            'fs_stability':nogueira_stability(recs, all_features),
            'n_outer_evaluations':len(g),
        })
    return pd.DataFrame(rows).sort_values('auprc_mean', ascending=False).reset_index(drop=True)


## Run Stage 1

Set one method in `STAGE1_METHODS` in the configuration cell, then run this section. Feature selection is refitted inside the nested cross-validation workflow.


In [ ]:
print("DMP Stage 1")
print("Method:", STAGE1_METHODS[0])
print("Ratios:", RATIOS)
print("Classifiers:", CLASSIFIER_NAMES)
print("Checkpoint:", CHECKPOINT_ROOT)

stage1_metrics, stage1_records, stage1_predictions = run_stage1(
    X, y, STAGE1_METHODS
)


## Save Stage 1 results


In [ ]:
from datetime import datetime

stage1_method = STAGE1_METHODS[0]
stage1_summary = summarize_benchmark(
    stage1_metrics, stage1_records, X.columns
)

run_label = datetime.now().strftime("%Y%m%d_%H%M%S")
stage1_run_dir = OUTPUT_ROOT / "stage1" / "runs" / run_label
stage1_run_dir.mkdir(parents=True, exist_ok=True)

stage1_metrics.to_csv(
    stage1_run_dir / f"stage1_{stage1_method}_outer_fold_metrics.csv",
    index=False,
)
stage1_summary.to_csv(
    stage1_run_dir / f"stage1_{stage1_method}_summary.csv",
    index=False,
)
stage1_predictions.to_csv(
    stage1_run_dir / f"stage1_{stage1_method}_outer_predictions.csv",
    index=False,
)

print("Saved to:", stage1_run_dir)


## Run Stage 2

Stage 1 is fixed before this benchmark. Stage 2 and classifier tuning remain inside nested cross-validation.


In [ ]:
LOCKED_STAGE1_METHOD = "relieff"
LOCKED_STAGE1_RATIO = 0.2

STAGE2_METHODS = ["lasso", "elasticnet", "rf"]
STAGE2_RATIOS = [0.2, 0.4, 0.6, 0.8, 1.0]

stage2_run_config = {
    "dataset": DATASET_NAME,
    "stage": "stage2",
    "stage1_method": LOCKED_STAGE1_METHOD,
    "stage1_ratio": LOCKED_STAGE1_RATIO,
    "stage2_methods": STAGE2_METHODS,
    "stage2_ratios": STAGE2_RATIOS,
    "classifiers": CLASSIFIER_NAMES,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "repeats": N_REPEATS,
    "base_seed": BASE_SEED,
    "param_grids": PARAM_GRIDS,
}

stage2_id = stage2_run_id(stage2_run_config)
stage2_root = OUTPUT_ROOT / "stage2" / "checkpoints" / stage2_id
stage2_checkpoint = AtomicStage2Checkpoint(stage2_root)
(stage2_root / "run_config.json").write_text(
    json.dumps(stage2_run_config, indent=2)
)

print("Stage 2 run ID:", stage2_id)
print("Checkpoint:", stage2_root)


In [ ]:
def make_stage2_pipeline(classifier_name, spec, seed):
    stage1_fs = RankedSelector(
        method=spec.stage1_method,
        ratio=spec.stage1_ratio,
        random_state=seed,
    )
    stage2_fs = RankedSelector(
        method=spec.stage2_method,
        ratio=spec.stage2_ratio,
        random_state=seed,
    )
    classifier_pipeline = make_classifier_pipeline(classifier_name, seed)

    return Pipeline(
        [("stage1_fs", stage1_fs), ("stage2_fs", stage2_fs)]
        + list(classifier_pipeline.steps),
        memory=ML_MEMORY,
    )


In [ ]:
for stage2_method in STAGE2_METHODS:
    for stage2_ratio in STAGE2_RATIOS:
        spec = Stage2Spec(
            stage1_method=LOCKED_STAGE1_METHOD,
            stage1_ratio=LOCKED_STAGE1_RATIO,
            stage2_method=stage2_method,
            stage2_ratio=stage2_ratio,
        )

        for classifier_name in CLASSIFIER_NAMES:
            print("[Stage 2]", spec.selector_label(), classifier_name)
            evaluate_stage2_configuration(
                X,
                y,
                spec=spec,
                classifier=classifier_name,
                pipeline_factory=make_stage2_pipeline,
                param_grid=PARAM_GRIDS[classifier_name],
                checkpoint=stage2_checkpoint,
                outer_folds=OUTER_FOLDS,
                inner_folds=INNER_FOLDS,
                repeats=N_REPEATS,
                base_seed=BASE_SEED,
                n_jobs=N_JOBS_GRID,
            )


## Save Stage 2 results


In [ ]:
stage2_paths = stage2_checkpoint.export()
stage2_metrics = pd.read_csv(stage2_paths["metrics"])

expected_rows = (
    len(STAGE2_METHODS)
    * len(STAGE2_RATIOS)
    * len(CLASSIFIER_NAMES)
    * N_REPEATS
    * OUTER_FOLDS
)
if len(stage2_metrics) != expected_rows:
    raise RuntimeError(
        f"Stage 2 is incomplete: expected {expected_rows} rows, "
        f"found {len(stage2_metrics)}."
    )

stage2_summary = summarize_stage2(stage2_metrics)
stage2_summary.to_csv(
    stage2_root / "stage2_outer_fold_summary.csv",
    index=False,
)
display(stage2_summary)
